# Modelo para el Analisis de factores que influyen en la popularidad de los videojuegos

Institucion: Escuela Politecnica Nacional EPN  
Materia: Metodos Numericos  
Semestre: 2026-A  
Estudiantes:  
* Ricardo Cisneros  
* Hugo Coyago  
* Jair Paez  

---

In [8]:
"""Importa las bibliotecas requeridas para el proyecto."""
import os
import csv
import re
import pandas as pd
import numpy as np

# Parte 1: Unificacion del Dataset
En esta etapa, reconstruimos el dataset a partir de los datos crudos extraidos de Steam y SteamSpy.

In [9]:
def analizar_rango_dueños(cadena_dueños: str) -> "int | str":
    """
    Calcula el punto medio del rango de dueños estimados.

    Args:
        cadena_dueños (str): Cadena de texto con el rango de dueños (ej. "10,000 .. 20,000").
        
    Returns:
        "int | str": El punto medio del rango como valor entero, o una cadena vacía en caso de error.
    """
    if not cadena_dueños or '..' not in cadena_dueños:
        return ""
    partes = cadena_dueños.split('..')
    if len(partes) == 2:
        try:
            limite_inferior = int(partes[0].replace(',', '').strip())
            limite_superior = int(partes[1].replace(',', '').strip())
            return (limite_inferior + limite_superior) // 2
        except ValueError:
            return ""
    return ""

def extraer_año_lanzamiento(fecha_str: str) -> str:
    """
    Extrae el año de lanzamiento desde una fecha dada en formato texto.

    Args:
        fecha_str (str): Cadena de texto que contiene la fecha de lanzamiento.

    Returns:
        str: El año extraído (ej. "2015"), o una cadena vacía si no se encuentra coincidencia.
    """
    if not fecha_str or fecha_str == "\\N":
        return ""
    coincidencia = re.search(r'\b(19\d\d|20\d\d)\b', fecha_str)
    if coincidencia:
        return coincidencia.group(1)
    return ""

def unificar_dataset() -> str:
    """
    Ejecuta la unificación de todos los datos en un solo conjunto de datos consolidados.

    Returns:
        str: La ruta del archivo CSV donde se guardaron los datos unificados.
    """
    print("Iniciando unificación de datos...")

    directorio_base = ".."
    juegos_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "games", "games.csv")
    reseñas_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "reviews", "reviews.csv")
    steamspy_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "steamspy_insights", "steamspy_insights.csv")
    categorias_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "categories", "categories.csv")
    salida_datos_unificados_crudos = os.path.join(directorio_base, "01_Datos", "steam_raw_unified_all_columns.csv")

    """Carga las categorías por aplicación."""
    datos_categorias = {}
    with open(categorias_csv, mode='r', encoding='utf-8') as archivo_cat:
        lector = csv.DictReader(archivo_cat)
        for fila in lector:
            id_app = fila.get("app_id")
            if id_app:
                datos_categorias[id_app] = datos_categorias.get(id_app, 0) + 1

    """Carga los datos principales de los juegos."""
    datos_juegos = {}
    with open(juegos_csv, mode='r', encoding='utf-8') as archivo_juegos:
        lector = csv.DictReader(archivo_juegos)
        for fila in lector:
            id_aplicacion = fila.get("app_id")
            if not id_aplicacion:
                continue
            
            idiomas_str = fila.get("languages", "").strip()
            cant_idiomas = 0
            if idiomas_str and idiomas_str != "\\N":
                cant_idiomas = len([lang.strip() for lang in idiomas_str.split(",") if lang.strip()])

            es_gratis_str = fila.get("is_free", "").strip().lower()
            es_gratis_num = 1 if es_gratis_str in ["true", "1"] else (0 if es_gratis_str in ["false", "0"] else "")

            datos_juegos[id_aplicacion] = {
                "name": fila.get("name", ""),
                "release_date": fila.get("release_date", ""),
                "release_year": extraer_año_lanzamiento(fila.get("release_date", "")),
                "is_free": es_gratis_num,
                "price_overview": fila.get("price_overview", ""),
                "games_languages": fila.get("languages", ""),
                "language_count": cant_idiomas,
                "type": fila.get("type", "")
            }

    """Carga las reseñas de los usuarios."""
    datos_reseñas = {}
    with open(reseñas_csv, mode='r', encoding='utf-8') as archivo_reseñas:
        lector = csv.DictReader(archivo_reseñas)
        for fila in lector:
            id_aplicacion = fila.get("app_id")
            if not id_aplicacion:
                continue
            datos_reseñas[id_aplicacion] = {
                "review_score": fila.get("review_score", ""),
                "review_score_description": fila.get("review_score_description", ""),
                "reviews_positive": fila.get("positive", ""),
                "reviews_negative": fila.get("negative", ""),
                "reviews_total": fila.get("total", ""),
                "metacritic_score": fila.get("metacritic_score", ""),
                "reviews_text": fila.get("reviews", ""),
                "recommendations": fila.get("recommendations", ""),
                "steamspy_user_score": fila.get("steamspy_user_score", ""),
                "steamspy_score_rank": fila.get("steamspy_score_rank", ""),
                "reviews_steamspy_positive": fila.get("steamspy_positive", ""),
                "reviews_steamspy_negative": fila.get("steamspy_negative", "")
            }

    """Carga las métricas de SteamSpy usando Pandas con soporte de escape de caracteres."""
    datos_steamspy = {}
    todos_los_ids_aplicacion = set(datos_juegos.keys()) | set(datos_reseñas.keys())
    
    df_spy_raw = pd.read_csv(steamspy_csv, dtype=str, escapechar='\\')
    for _, fila_spy in df_spy_raw.iterrows():
        fila = fila_spy.to_dict()
        id_aplicacion = fila.get("app_id")
        if not id_aplicacion:
            continue
        datos_steamspy[id_aplicacion] = fila
        todos_los_ids_aplicacion.add(id_aplicacion)

    """Unifica los datos en un solo conjunto."""
    todos_los_datos_fusionados = []
    for id_aplicacion in sorted(list(todos_los_ids_aplicacion), key=lambda x: int(x) if x.isdigit() else 9999999):
        juego = datos_juegos.get(id_aplicacion, {
            "name": "", "release_date": "", "release_year": "", "is_free": "", "price_overview": "",
            "games_languages": "", "language_count": "", "type": ""
        })
        reseña = datos_reseñas.get(id_aplicacion, {
            "review_score": "", "review_score_description": "", "reviews_positive": "", "reviews_negative": "",
            "reviews_total": "", "metacritic_score": "", "reviews_text": "", "recommendations": "",
            "steamspy_user_score": "", "steamspy_score_rank": "", "reviews_steamspy_positive": "", "reviews_steamspy_negative": ""
        })
        spy = datos_steamspy.get(id_aplicacion, {
            "developer": "", "publisher": "", "owners_range": "", "concurrent_users_yesterday": "",
            "playtime_average_forever": "", "playtime_average_2weeks": "", "playtime_median_forever": "",
            "playtime_median_2weeks": "", "price": "", "initial_price": "", "discount": "", "languages": "", "genres": ""
        })

        """Convierte los precios a dólares."""
        precio_dolares = ""
        if spy.get("price", "") and spy.get("price", "") != "\\N":
            try:
                precio_dolares = round(int(spy.get("price", "")) / 100.0, 2)
            except ValueError:
                pass

        precio_inicial_dolares = ""
        if spy.get("initial_price", "") and spy.get("initial_price", "") != "\\N":
            try:
                precio_inicial_dolares = round(int(spy.get("initial_price", "")) / 100.0, 2)
            except ValueError:
                pass

        """Calcula la proporción de aprobación."""
        proporcion_aprobacion = ""
        if reseña.get("reviews_positive", "") and reseña.get("reviews_negative", ""):
            try:
                positivas = int(reseña.get("reviews_positive", ""))
                negativas = int(reseña.get("reviews_negative", ""))
                if positivas + negativas > 0:
                    proporcion_aprobacion = round(positivas / (positivas + negativas), 4)
            except ValueError:
                pass

        """Genera la fila unificada."""
        fila_fusionada = {
            "app_id": id_aplicacion,
            "name": juego["name"],
            "release_date": juego["release_date"],
            "release_year": juego["release_year"],
            "is_free": juego["is_free"],
            "price_overview": juego["price_overview"],
            "games_languages": juego["games_languages"],
            "language_count": juego["language_count"],
            "category_count": datos_categorias.get(id_aplicacion, 0),
            "type": juego["type"],
            "developer": spy.get("developer", ""),
            "publisher": spy.get("publisher", ""),
            "owners_range": spy.get("owners_range", ""),
            "concurrent_users_yesterday": spy.get("concurrent_users_yesterday", ""),
            "playtime_average_forever": spy.get("playtime_average_forever", ""),
            "playtime_average_2weeks": spy.get("playtime_average_2weeks", ""),
            "playtime_median_forever": spy.get("playtime_median_forever", ""),
            "playtime_median_2weeks": spy.get("playtime_median_2weeks", ""),
            "steamspy_price": spy.get("price", ""),
            "steamspy_initial_price": spy.get("initial_price", ""),
            "steamspy_discount": spy.get("discount", ""),
            "steamspy_languages": spy.get("languages", ""),
            "genres": spy.get("genres", ""),
            "review_score": reseña["review_score"],
            "review_score_description": reseña["review_score_description"],
            "reviews_positive": reseña["reviews_positive"],
            "reviews_negative": reseña["reviews_negative"],
            "reviews_total": reseña["reviews_total"],
            "metacritic_score": reseña["metacritic_score"],
            "reviews_text": reseña["reviews_text"],
            "recommendations": reseña["recommendations"],
            "steamspy_user_score": reseña["steamspy_user_score"],
            "steamspy_score_rank": reseña["steamspy_score_rank"],
            "reviews_steamspy_positive": reseña["reviews_steamspy_positive"],
            "reviews_steamspy_negative": reseña["reviews_steamspy_negative"],
            "price_usd": precio_dolares,
            "steamspy_initial_price_usd": precio_inicial_dolares,
            "approval_ratio": proporcion_aprobacion,
            "estimated_owners": analizar_rango_dueños(spy.get("owners_range", ""))
        }
        todos_los_datos_fusionados.append(fila_fusionada)

    """Exporta los datos a un archivo CSV."""
    encabezados = list(todos_los_datos_fusionados[0].keys())
    with open(salida_datos_unificados_crudos, mode='w', newline='', encoding='utf-8') as archivo_salida:
        escritor = csv.DictWriter(archivo_salida, fieldnames=encabezados)
        escritor.writeheader()
        escritor.writerows(todos_los_datos_fusionados)

    print("Consolidación completada con éxito.")
    return salida_datos_unificados_crudos

# Ejecución de la función principal de la Parte 1
archivo_unificado = unificar_dataset()

Iniciando unificación de datos...
Consolidación completada con éxito.


# Parte 2: Limpieza de Datos

Proceso de limpieza y tratamiento de nulos.

In [10]:
def limpiar_datos(ruta_archivo: str) -> "tuple[pd.DataFrame, list[str]]":
    """
    Limpia los datos, procesa nulos y filtra variables independientes de alta calidad.

    Args:
        ruta_archivo (str): Ruta al archivo CSV con los datos crudos unificados.

    Returns:
        "tuple[pd.DataFrame, list[str]]": Un DataFrame limpio y la lista de variables clave.
    """
    df = pd.read_csv(ruta_archivo, dtype={'app_id': str}, low_memory=False)

    """Define las variables clave del modelo predictivo."""
    variable_objetivo = 'estimated_owners'
    variables_predictoras = [
        'price_usd', 'is_free', 'approval_ratio', 'reviews_total',
        'language_count', 'category_count', 'release_year'
    ]
    todas_las_variables = [variable_objetivo] + variables_predictoras

    """Limpia los registros duplicados."""
    df = df.drop_duplicates(subset=['app_id'])

    """Convierte los valores nulos a NaN y elimina las filas incompletas."""
    for col in todas_las_variables:
            df[col] = df[col].replace([r'\\N', 'N/A', 'nan', 'NaN', ''], np.nan)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df_limpio = df.dropna(subset=todas_las_variables).copy()

    """Filtra los valores incoherentes (precios negativos, variables fuera de rango)."""
    df_limpio = df_limpio[
        (df_limpio['price_usd'] >= 0) &
        (df_limpio['is_free'].isin([0.0, 1.0, 0, 1])) &
        (df_limpio['reviews_total'] >= 0) &
        (df_limpio['language_count'] >= 0) &
        (df_limpio['category_count'] >= 0) &
        (df_limpio['release_year'] >= 1997) & (df_limpio['release_year'] <= 2024) &
        (df_limpio['approval_ratio'] >= 0.0) & (df_limpio['approval_ratio'] <= 1.0) &
        (df_limpio['estimated_owners'] > 0)
    ]
    
    """Filtra el dataset para conservar unicamente las 8 variables clave del modelo."""
    df_limpio = df_limpio[todas_las_variables]
    
    print("Limpieza de datos completada.")
    return df_limpio, todas_las_variables

# Ejecución de la función principal de la Parte 2
df_limpio, todas_las_variables = limpiar_datos(archivo_unificado)

Limpieza de datos completada.


# Parte 3: Normalizacion de Datos

Escalamiento de variables (Z-Score).

In [11]:
def normalizar_datos(df_limpio: pd.DataFrame, todas_las_variables: "list[str]") -> "tuple[pd.DataFrame, list[str]]":
    """
    Normaliza los datos numéricos y exporta el conjunto final a un CSV.

    Args:
        df_limpio (pd.DataFrame): DataFrame con los datos previamente limpiados.
        todas_las_variables ("list[str]"): Lista de las variables clave del dataset.

    Returns:
        "tuple[pd.DataFrame, list[str]]": El DataFrame con variables normalizadas y la lista de variables numéricas procesadas.
    """
    """Aplica la transformación logarítmica a reviews_total para reducir el sesgo extremo y alinearse con Grm y Šubelj (2021)."""
    if 'reviews_total' in df_limpio.columns:
        df_limpio['reviews_total'] = np.log10(df_limpio['reviews_total'] + 1)

    """Normaliza las variables usando el metodo Z-score (estandarizacion con media y desviacion estandar)."""
    variables_numericas = todas_las_variables
    for col in variables_numericas:
        mean_val = df_limpio[col].mean()
        std_val = df_limpio[col].std()
        df_limpio[col + '_normalized'] = (df_limpio[col] - mean_val) / std_val if std_val > 0.0 else 0.0

    """Exporta las variables del modelo limpio a un archivo CSV."""
    directorio_base = ".."
    archivo_salida = os.path.join(directorio_base, "01_Datos", "steam_cleaned_data.csv")
    df_limpio[todas_las_variables].to_csv(archivo_salida, index=False)
    
    print("Normalización completada y exportada con éxito.")
    return df_limpio, variables_numericas

# Ejecución de la función principal de la Parte 3
df_final, variables_numericas = normalizar_datos(df_limpio, todas_las_variables)
df_final.head()

Normalización completada y exportada con éxito.


,estimated_owners,price_usd,is_free,approval_ratio,reviews_total,language_count,category_count,release_year,estimated_owners_normalized,price_usd_normalized,is_free_normalized,approval_ratio_normalized,reviews_total_normalized,language_count_normalized,category_count_normalized,release_year_normalized
0,15000000.0,9.99,0.0,0.9743,5.383117,1.0,6,2000.0,14.996682,0.164236,-0.328223,0.930332,4.170313,-0.103642,0.627522,-6.263225
1,7500000.0,4.99,0.0,0.8699,3.924796,1.0,7,1999.0,7.448984,-0.235566,-0.328223,0.497153,2.577329,-0.103642,0.999207,-6.576817
2,7500000.0,4.99,0.0,0.9029,3.840232,1.0,3,2003.0,7.448984,-0.235566,-0.328223,0.634077,2.484956,-0.103642,-0.487533,-5.322448
3,7500000.0,4.99,0.0,0.8291,3.486714,1.0,7,2001.0,7.448984,-0.235566,-0.328223,0.327864,2.098794,-0.103642,0.999207,-5.949633
4,3500000.0,4.99,0.0,0.9525,4.368752,1.0,5,1999.0,3.423544,-0.235566,-0.328223,0.839879,3.062280,-0.103642,0.255837,-6.576817
